In [0]:
USE CATALOG biking_product_sales_lakehouse;
USE SCHEMA gold

In [0]:
CREATE OR REPLACE TABLE dim_date AS
WITH date_sequence AS (
    SELECT 
        -- Generate date sequence based on order dates
        EXPLODE(SEQUENCE(
            (SELECT MIN(CAST(order_date AS DATE)) FROM silver_curated.sales),
            (SELECT MAX(CAST(order_date AS DATE)) FROM silver_curated.sales),
            INTERVAL 1 DAY
        )) AS full_date
)
SELECT
    INT(DATE_FORMAT(full_date, 'yyyyMMdd')) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month,
    DATE_FORMAT(full_date, 'MMMM') AS month_name,
    DAY(full_date) AS day_of_month,
    DAYOFWEEK(full_date) AS day_of_week,
    DATE_FORMAT(full_date, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM date_sequence;

In [0]:
-- Customers from crm
CREATE OR REPLACE TABLE dim_customer AS
SELECT
    customer_key,
    customer_id,
    firstname,
    lastname,
    CONCAT(firstname, ' ', lastname) AS full_name,
    gender,
    marital_status,
    country,
    creation_date
FROM silver_curated.customer_crm;

In [0]:
CREATE OR REPLACE TABLE dim_product AS
WITH deduplicated_categories AS (
    -- Group or deduplicate category mapping to prevent row fan-out
    SELECT 
        category_id,
        category,
        sub_category,
        maintenance,
        ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY category_id) AS rn
    FROM silver_curated.product_category
)
SELECT
    p.product_key,
    p.product_id,
    p.product_name,
    p.product_cost,
    p.product_start_date,
    p.product_end_date,
    p.product_line,
    c.category,
    c.sub_category,
    c.maintenance
FROM silver_curated.products p
LEFT JOIN deduplicated_categories c 
       ON p.product_category_id = c.category_id AND c.rn = 1;

In [0]:
CREATE OR REPLACE TABLE fact_sales AS
SELECT
    s.order_id,
    -- Dimension Keys
    s.customer_id,
    s.product_key,
    -- Degenerate / Metadata Attributes
    s.order_date,
    s.ship_date,
    s.due_date,
    -- Facts / Metrics
    s.quantity,
    s.price,
    s.sales
FROM silver_curated.sales s;